In [1]:
# importing different modules
# numpy is for math and matrix/vector stuff
# matplotlib is for visualizing data
# PIL allows for opening image files
# torchvision and torch are for ML handling
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import torchvision.models as models
import torch

# os and sys allow us to do stuff with files
import os
import sys

In [2]:
## taken from pytorch forums
# creates an array of class labels where idx2label[i] is the string label associated with class i
import json
class_idx = json.load(open("imagenet_class_index.json"))
idx2label = np.array([class_idx[str(k)][1] for k in range(len(class_idx))])
#print(idx2label)
def convertIndexToLabel(i):
    return idx2label[i]

# using the imagenet official correct validation classifications, turns those class numbers into a list of classification names
# these are the "correct" classes for each image in the dataset
file_path = 'val.txt'
true_class_nums = np.loadtxt(file_path, dtype = int)
true_class_names = convertIndexToLabel(true_class_nums)

In [3]:

# Defining the different models, using pretrained weights
alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
resnet = models.resnet34(weights = models.ResNet34_Weights.DEFAULT)
convnext = models.convnext_base(weights = models.ConvNeXt_Base_Weights.DEFAULT)


# Setting the models to evaluation mode
alexnet.eval()
resnet.eval()
convnext.eval()

# Get the preprocessing transformations necessary in order for the models to take the image inputs
alex_preprocess = models.AlexNet_Weights.DEFAULT.transforms()
res_preprocess = models.ResNet34_Weights.DEFAULT.transforms()
conv_preprocess = models.ConvNeXt_Base_Weights.DEFAULT.transforms()


# setup that allows us to iterate through the different models
class modelinfo:
    def __init__(self, model, preprocess):
        self.name = model._get_name()
        self.model = model
        self.preprocess = preprocess
alexinfo = modelinfo(alexnet, alex_preprocess)
resinfo = modelinfo(resnet, res_preprocess)
convinfo = modelinfo(convnext, conv_preprocess)
used_models = [alexinfo, resinfo, convinfo]

In [4]:
print(alexnet._get_name())

AlexNet


In [5]:
file_path = os.path.join(os.getcwd(), "imagenet_val_dataset")
files = os.listdir(file_path)
files.sort()

# params to determine which slice of the validation set (or whatever set we're using) we're looking at
start = 0
num_iter = 100

# iterates through the models, and for each model, will run all the images from start to start+num_iter through the model
# it will then add the labels, and images (as numpy array) to associated arrays for further analysis
def inference(files, used_models, num_iter, start = 0):
    model_labels = []
    model_np_imgs = []
    flag = True
    for model in used_models:
        print(model.model._get_name())
        labels = []
        for i, file in enumerate(files[start:min(len(files),start+num_iter)]):
            # code for making countdown
            print(f"\r{i+1}/{num_iter}", end="")
            sys.stdout.flush()

            # running the file through the models
            if file[-5:] == ".JPEG":
                img = Image.open("imagenet_val_dataset/"+file)
                #img.convert("RGB") is important because some images are in black-and-white,
                #which means they have a different tensor size as colored images, since they 
                #are missing a color channel
                input = model.preprocess(img.convert("RGB")).unsqueeze(0)
                # gets the "probability" for each class
                prob = torch.nn.functional.softmax(model.model(input)[0], dim=0)
                # sorts the class labels in order from highest to lowest "probability"
                sorted_prob_label = sorted(zip(prob, idx2label), reverse = True)
                sorted_prob, sorted_label = zip(*list(sorted_prob_label))
                # gets the predicted label
                label = sorted_label[0]
                labels.append(label)
                # adds a copy of the image so we can display alongside data below
                if(flag):
                    img_np = np.array(img)
                    model_np_imgs.append(img_np)
        model_labels.append(labels)
        flag = False
        print("")
    return model_labels

In [6]:
model_labels = inference(files, used_models, num_iter, start)

AlexNet
100/100
ResNet
100/100
ConvNeXt
100/100


In [7]:
def get_misclassified(used_models, model_labels, dest_directory = "misclassified_images_by_class/", true_class_names = true_class_names):
    misclass_pics = []
    misclass_count = []
    for j in range(len(model_labels)):
        misclass_count.append(0)
        misclass_pics.append([])
        with open(dest_directory+used_models[j].name,'w') as file:
            for i, label in enumerate(model_labels[j]):
                truelabel = true_class_names[i+start]
                if label != truelabel:
                    misclass_count[j] += 1
                    num_zeros = 7
                    count = i+1
                    for n in range(1,9):
                        #print(count)
                        if count // 10 != 0:
                            count = count // 10
                            num_zeros -= 1
                        else:
                            break
                    #print(num_zeros)
                    image_name = "ILSVRC2012_val_%s%d.JPEG" % (num_zeros*"0", i+1)
                    misclass_pics[j].append(image_name)
                    file.write(f"{image_name}\n")
    print(misclass_pics)
    print(misclass_count)
    #accuracy
    total_classified = len(model_labels[0])
    accuracy = ((np.zeros(3)+total_classified)-np.array(misclass_count))/total_classified
    print(accuracy)
    
    return misclass_pics, misclass_count, accuracy


In [8]:
## NOTE: THIS WILL REPLACE THE CONTENTS OF THE FILES EACH TIME! USE ALTERNATE DIRECTORY OR CHANGE CODE IF YOU DONT WANT THIS TO HAPPEN
misclass_pics, misclass_count, accuracy = get_misclassified(used_models, model_labels)

[['ILSVRC2012_val_00000001.JPEG', 'ILSVRC2012_val_00000002.JPEG', 'ILSVRC2012_val_00000004.JPEG', 'ILSVRC2012_val_00000005.JPEG', 'ILSVRC2012_val_00000006.JPEG', 'ILSVRC2012_val_00000008.JPEG', 'ILSVRC2012_val_00000009.JPEG', 'ILSVRC2012_val_00000010.JPEG', 'ILSVRC2012_val_00000017.JPEG', 'ILSVRC2012_val_00000018.JPEG', 'ILSVRC2012_val_00000019.JPEG', 'ILSVRC2012_val_00000022.JPEG', 'ILSVRC2012_val_00000028.JPEG', 'ILSVRC2012_val_00000030.JPEG', 'ILSVRC2012_val_00000033.JPEG', 'ILSVRC2012_val_00000037.JPEG', 'ILSVRC2012_val_00000039.JPEG', 'ILSVRC2012_val_00000040.JPEG', 'ILSVRC2012_val_00000041.JPEG', 'ILSVRC2012_val_00000044.JPEG', 'ILSVRC2012_val_00000045.JPEG', 'ILSVRC2012_val_00000048.JPEG', 'ILSVRC2012_val_00000050.JPEG', 'ILSVRC2012_val_00000055.JPEG', 'ILSVRC2012_val_00000059.JPEG', 'ILSVRC2012_val_00000061.JPEG', 'ILSVRC2012_val_00000062.JPEG', 'ILSVRC2012_val_00000063.JPEG', 'ILSVRC2012_val_00000064.JPEG', 'ILSVRC2012_val_00000065.JPEG', 'ILSVRC2012_val_00000069.JPEG', 'ILSVR